In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@usgridenergypipeline.dfs.core.windows.net/"
SILVER_PATH = "abfss://silver@usgridenergypipeline.dfs.core.windows.net/silver_region_data"
CHECKPOINT  = "abfss://bronze@usgridenergypipeline.dfs.core.windows.net/_checkpoints/region-data"

DIM_METRIC_PATH = "abfss://silver@usgridenergypipeline.dfs.core.windows.net/dim_metric_type"
DIM_REGION_PATH = "abfss://silver@usgridenergypipeline.dfs.core.windows.net/dim_region"

SILVER_TABLE = "us_grid_energy_pipeline_databricks.usgrid.silver_region_data"

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS usgrid")

DataFrame[]

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        region_code STRING,
        metric_type_code STRING,
        period TIMESTAMP,
        value_in_megawatts FLOAT
        
    )
    USING DELTA
    LOCATION '{SILVER_PATH}'
    PARTITIONED BY (region_code, metric_type_code)
    """
    )

DataFrame[]

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS usgrid.dim_metric_type (
        metric_type_code STRING,
        metric_type_name STRING
    )
    USING DELTA
    LOCATION '{DIM_METRIC_PATH}'
""")

DataFrame[]

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS usgrid.dim_region (
        region_code STRING,
        region_name STRING
    )
    USING DELTA
    LOCATION '{DIM_REGION_PATH}'
""")

DataFrame[]

In [0]:
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("multiLine", "true")
    .load(BRONZE_PATH + "*/region-data/*/*/*.json")
)

In [0]:
df_exploded = (
    df_raw
    .withColumn("facet_block", F.explode("raw_data"))
    .withColumn("row", F.explode("facet_block.facet_data.response.data"))
    .select(
        F.col("region").alias("region_code"),
        F.col("facet_block.facet_type").alias("metric_type_code"),
        F.col("facet_block.facet_name").alias("metric_type_name"),
        F.col("row.respondent-name").alias("region_name"),
        F.col("row.period").alias("period"),
        F.col("row.value").alias("value_in_megawatts"),
    )
)

In [0]:
df_typed = (
    df_exploded
    .withColumn("period",F.to_timestamp("period", "yyyy-MM-dd'T'HH"))
    .withColumn("value_in_megawatts", F.col("value_in_megawatts").cast("float"))
    .dropDuplicates(["region_code", "metric_type_code", "period"])
)

In [0]:
def merge_to_silver(micro_batch_df, batch_id):

    # 1. Dim metric type table
    dim_metric_df = (
        micro_batch_df
        .select("metric_type_code", "metric_type_name")
        .dropDuplicates(["metric_type_code"])
    )
    dim_metric_df.createOrReplaceTempView("dim_metric_updates")

    spark.sql("""
        MERGE INTO us_grid_energy_pipeline_databricks.usgrid.dim_metric_type AS target
        USING dim_metric_updates AS source
        ON target.metric_type_code = source.metric_type_code
        WHEN NOT MATCHED THEN INSERT *
    """)

    # 2. Dim region table
    dim_region_df = (
        micro_batch_df
        .select("region_code", "region_name")
        .dropDuplicates(["region_code"])
    )
    dim_region_df.createOrReplaceTempView("dim_region_updates")

    spark.sql("""
        MERGE INTO us_grid_energy_pipeline_databricks.usgrid.dim_region AS target
        USING dim_region_updates AS source
        ON target.region_code = source.region_code
        WHEN NOT MATCHED THEN INSERT *
    """)

    # 3. Silver table
    silver_df = micro_batch_df.drop("metric_type_name", "region_name")
    silver_df.createOrReplaceTempView("silver_updates")

    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING silver_updates AS source
        ON target.region_code = source.region_code
        AND target.metric_type_code = source.metric_type_code
        AND target.period = source.period
        WHEN MATCHED AND target.value_in_megawatts != source.value_in_megawatts
            THEN UPDATE SET target.value_in_megawatts = source.value_in_megawatts
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
(
    df_typed
    .writeStream
    .format("delta")
    .foreachBatch(merge_to_silver)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [0]:
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.dim_metric_type")
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.dim_region")

In [0]:
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.silver_region_data")

In [0]:
# dbutils.fs.rm("abfss://silver-dev@usgridenergypipeline.dfs.core.windows.net/silver_region-data", recurse=True)

In [0]:
# dbutils.fs.rm("abfss://bronze@usgridenergypipeline.dfs.core.windows.net/_checkpoints/fuel_type_data", recurse=True)
# dbutils.fs.rm("abfss://bronze@usgridenergypipeline.dfs.core.windows.net/_checkpoints/region-data", recurse=True)

In [0]:
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data")
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.silver_region_data")
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.dim_fuel_type")
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.dim_metric_type")
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.dim_region")

In [0]:
# dbutils.fs.rm("abfss://silver@usgridenergypipeline.dfs.core.windows.net/silver_fuel_type_data", recurse=True)
# dbutils.fs.rm("abfss://silver@usgridenergypipeline.dfs.core.windows.net/silver_region_data", recurse=True)
# dbutils.fs.rm("abfss://silver@usgridenergypipeline.dfs.core.windows.net/dim_fuel_type", recurse=True)
# dbutils.fs.rm("abfss://silver@usgridenergypipeline.dfs.core.windows.net/dim_metric_type", recurse=True)
# dbutils.fs.rm("abfss://silver@usgridenergypipeline.dfs.core.windows.net/dim_region", recurse=True)

In [0]:
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.gold_fuel_metrics")
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.gold_region_metrics")

In [0]:
# dbutils.fs.rm("abfss://gold@usgridenergypipeline.dfs.core.windows.net/gold_fuel_metrics", recurse=True)
# dbutils.fs.rm("abfss://gold@usgridenergypipeline.dfs.core.windows.net/gold_region_metrics", recurse=True)